# Обучение ИИ-детектора v2 (XLM-R / mDeBERTa → ONNX)

Среда: **Runtime → Change runtime type → GPU (T4/L4)**.

Пайплайн v2:
1. `make_dataset_v2.py` — баланс `ru` / `uz-Latn` / `uz-Cyrl`, split по `doc_id`
2. `gen_ai_v2.py` — парный rewrite/continue (опционально)
3. `train.py` — метрики по языкам + temperature scaling
4. `export_onnx.py` — ONNX + `calibration.json`


## 1. Код проекта


In [ ]:
!git clone https://github.com/13unethical/antiplagiat.git || true
%cd antiplagiat/backend/ml
!pip -q install -r requirements.txt


## 2. Данные

Загрузите **`dataset_v2.zip`** (`train.jsonl` + `val.jsonl` + `test.jsonl`)  
или соберите на месте из корпуса + `data/ai` / `data/ai_v2`.


In [ ]:
from google.colab import files
up = files.upload()  # выберите dataset_v2.zip
!mkdir -p data/v2 && unzip -o dataset_v2.zip -d data/v2
!wc -l data/v2/train.jsonl data/v2/val.jsonl data/v2/test.jsonl


## 3. Обучение (XLM-RoBERTa base)

Альтернатива: `--model microsoft/mdeberta-v3-base` (точнее, тяжелее).


In [ ]:
!python train.py --data data/v2 --model xlm-roberta-base --epochs 3 --batch 16 --out runs/ai-v2
!cat runs/ai-v2/calibration.json


## 4. Оценка


In [ ]:
!python eval.py --model runs/ai-v2 --data data/v2/val.jsonl
!python eval.py --model runs/ai-v2 --data data/v2/test.jsonl


## 5. Экспорт в ONNX + calibration.json


In [ ]:
!python export_onnx.py --model runs/ai-v2 --out export_v2
!cd export_v2 && zip -r ../ai_model_v2.zip . >/dev/null && echo OK
from google.colab import files; files.download('ai_model_v2.zip')


## 6. Подключение к бэкенду

1. Рабочую v1 не трогать: она в `backend/app/ml_model_v1_backup/`.
2. Распакуйте `ai_model_v2.zip` в `backend/app/ml_model/` **только после** проверки на val/test.
3. В `.env`: `AI_MODEL_DIR=./app/ml_model` (бэкенд читает `calibration.json` и делит logits на T).
